# Knee Bone Age — GPU training

Run top to bottom. Set **Runtime → Change runtime type → GPU** first.

Local CPU runs plateaued at ~2.2 years MAE because they were limited to 16x96x96 for
12-15 epochs. This trains at 32x192x192 (0.5 mm pixels, so a 1-2 mm growth plate is
actually resolvable) for 40 epochs.

In [ ]:
!nvidia-smi
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 1. Get the code
!git clone https://github.com/anshppatel4-crypto/knee-bone-age-ai.git
%cd knee-bone-age-ai
!pip install -q monai pydicom scikit-image

In [ ]:
# 2. Generate phantoms in parallel (~15-25 min for 400).
#    More data is the cheapest remaining win: 400 beats the 240 used locally.
import subprocess
jobs = [subprocess.Popen(['python', '-m', 'src.knee_phantom', '--count', '100',
                          '--out', f'data/phantom_gpu_{tag}', '--seed', str(seed),
                          '--slices', '32', '--resolution', '256'])
        for tag, seed in zip('abcd', [31, 32, 33, 34])]
for job in jobs:
    job.wait()
!ls -d data/phantom_gpu_*/knee_* | wc -l

In [ ]:
# 3. Train. Drop --batch-size to 2 if you hit CUDA out-of-memory.
!python src/train.py --data 'data/phantom_gpu_*'     --arch resnet34 --epochs 40 --batch-size 4 --lr 3e-4     --input-shape 32 192 192 --output final_knee_model_gpu.pth

In [ ]:
# 4. Did it actually learn? Two checks that matter more than the headline number.
import numpy as np, sys
sys.path.insert(0, '.')
from torch.utils.data import DataLoader
from src.model import load_checkpoint
from src.predict import predict_scan
from src.dataset import KneeVolumeDataset
from src.train import build_catalog, split_catalog, predict_loader, regression_metrics

test = split_catalog(build_catalog(['data/phantom_gpu_*']), seed=0)[2]
model, meta = load_checkpoint('final_knee_model_gpu.pth', 'cuda')
dataset = KneeVolumeDataset(test, input_shape=tuple(meta['input_shape']), augment=False)

pred, targ = predict_loader(model, DataLoader(dataset, batch_size=4), 'cuda', tta=True)
print(regression_metrics(pred, targ))

# Sex must matter: girls mature ~1.8 y earlier, so flipping sex should move the
# prediction 1.5-2 y. Local runs never got past 0.06 y, which meant sex was ignored.
deltas = []
for i in range(min(15, len(dataset))):
    volume = dataset[i]['image'].numpy()[0]
    deltas.append(predict_scan(model, volume, 'm', 'cuda')['bone_age']
                  - predict_scan(model, volume, 'f', 'cuda')['bone_age'])
print('sex delta (M-F):', round(float(np.mean(deltas)), 3), 'years')

In [ ]:
# 5. Download the checkpoint and metrics
from google.colab import files
files.download('final_knee_model_gpu.pth')
files.download('final_knee_model_gpu_metrics.json')

## Reading the result

- **Test MAE vs baseline**: training prints the "always guess the mean age" baseline. Beating it
  by a wide margin is the minimum bar.
- **Sex delta near 1.8**: the model is using skeletal maturity plus sex, as a radiologist does.
  Still near zero means it is underfitting again — train longer before believing the MAE.
- **Train loss well below val MAE**: finally fitting. Locally they were equal, which is underfitting.

Even a strong score here is accuracy **on synthetic phantoms**, not on patients.